**Library Import**

In [167]:
# os - to manage folders and files
import os
# numpy - used for arrays and numerical calculation
import numpy as np
# matplotlib - used to visualize graphs, images, and confusion matrices
import matplotlib.pyplot as plt
# tensorflow - used to train deep learning models
import tensorflow as tf
# keras - used for layers and models of keras
from tensorflow.keras import layers, models
# import early stopping library
from tensorflow.keras.callbacks import EarlyStopping
#sklearn - used to calculate performance for model evaluation tools from skelearn metric
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# To recognize dataset path**

In [169]:
#to set dataset path
DATASET_DIR = "/content/drive/MyDrive/apple_orange_dataset"

#to set training data path
TRAIN_DIR = os.path.join(DATASET_DIR, "train")

#to set velidation data path
VAL_DIR = os.path.join(DATASET_DIR, "validation")

#to check paths
print("Dataset Path : ")
print(DATASET_DIR)

print("\n Training Path : ")
print(TRAIN_DIR)

print("\n Validation Path : ")
print(VAL_DIR)

Dataset Path : 
/content/drive/MyDrive/apple_orange_dataset

 Training Path : 
/content/drive/MyDrive/apple_orange_dataset/train

 Validation Path : 
/content/drive/MyDrive/apple_orange_dataset/validation


**To check dataset folder**

In [170]:
#to check dataset folder
if not os.path.exists(TRAIN_DIR):
  raise FileNotFoundError()

#to check validation foler in dataset
if not os.path.exists(VAL_DIR):
  raise FileNotFoundError()

#to check what exist in train folder
print("Training Classes : ")
print(os.listdir(TRAIN_DIR))

#to check what exist in validation folder
print("Validation Classes : ")
print(os.listdir(VAL_DIR))


Training Classes : 
['apple', 'orange']
Validation Classes : 
['apple', 'orange']


**Count Images**

In [171]:
#to create a function to count images in each folder
def count_image(folder):
  count = 0
  #to find files in folder
  for root,dirs, files in os.walk(folder):
    #to check each file
    for file in files:
      #to check image file extension
      if file.lower().endswith((".jpg", ".png", ".jpeg", ".jfif")):
        count+=1
  return count

#apple training
apple_train = count_image(os.path.join(TRAIN_DIR, "apple"))

#orange training
orange_train = count_image(os.path.join(TRAIN_DIR, "orange"))

#apple valiaation
apple_val = count_image(os.path.join(VAL_DIR, "apple"))

#orange validation
orange_val = count_image(os.path.join(VAL_DIR, "orange"))

print("=====Dataset Information=====")
print("Apple Training Images : ", apple_train)
print("Orange Training Images : ", orange_train)
print("Apple Validation Images : ", apple_val)
print("Apple Validation Images : ", orange_val)

=====Dataset Information=====
Apple Training Images :  30
Orange Training Images :  30
Apple Validation Images :  10
Apple Validation Images :  10


**Display Sample Images**


In [ ]:
#to find apple images from apple folder
apple_files = [os.path.join(TRAIN_DIR, "apple", file) for file in os.listdir(os.path.join(TRAIN_DIR, "apple")) if file.lower().endswith((".jpg", ".png", ".jpeg", ".jfif"))]

#to find orange images from orange foler
orange_files = [os.path.join(TRAIN_DIR, "orange", file) for file in os.listdir(os.path.join(TRAIN_DIR, "orange")) if file.lower().endswith((".jpg", ".png", ".jpeg", ".jfif"))]

#choose 4 images
sample_apple = apple_files[:4]
sample_orange = orange_files[:4]

#combine apple and orange
sample_files = sample_orange + sample_apple

#set image size
plt.figure(figsize=(12, 8))

#display each images
for i, file in enumerate(sample_files):
  #read image
  from PIL import Image
  image = Image.open(file)

  #change 2 rows*4 columns
  plt.subplot(2, 4, i+1)

  #display image
  plt.imshow(image)

  #set label apple or orange
  if "/apple/" in file:
    label = "Apple"
  else:
    label = "Orange"
  plt.title(label)

  #don't show axis
  plt.axis("off")

  #main title
  plt.suptitle("Apple and Orange Training Images", fontsize = (20))

  #show graph
  # plt.show()

**Load Image Dataset**

- to set image size (150*150) pixels
- to set batch size (2)


In [ ]:
from sklearn.utils import shuffle
#change image size
IMG_SIZE = (150,150)

#set batch size in 2 total 10
BATCH_SIZE = 2

#training dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    #training
    TRAIN_DIR,

    #image size
    image_size = IMG_SIZE,

    #batch
    batch_size = BATCH_SIZE,

    #to make random order for training
    shuffle = True,

    #to set seed for equal randomization
    seed = 42
)

#validation dataset
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    #training
    VAL_DIR,

    #image size
    image_size = IMG_SIZE,

    #batch
    batch_size = BATCH_SIZE,

    #don't make random order for evalutation
    shuffle = False,
)

#to get class names from dataset
class_names = train_dataset.class_names

print("Class Names : ")
print(class_names)



**To Build CNN Model**

In [ ]:
#to build CNN (Convolutional Neural Network) model

model = models.Sequential([
    #input layer(image size=150, RGB image=3 channel)
    layers.Input(shape=(150, 150, 3)),

    #image rescaling(image pixels - from 0 to 255)
    #if 0, 0/255 = 0.0, if 255, 255/255 = 1.0
    #change value range from 0 to 1
    layers.Rescaling(1/255),

    #Convolutional layer 1
    #to find edge, shape, texture as feature from image
    layers.Conv2D(16,(3,3), activation="relu"),

    #max pooling 1 - to reduce size of feature map and computation time
    layers.MaxPool2D(),

    #Convolutional layer 2
    #to find edge, shape, texture as feature from image
    layers.Conv2D(32,(3,3), activation="relu"),

    #max pooling 2 - to reduce size of feature map and computation time
    layers.MaxPool2D(),

    #flatten - to change from 2D feature map to 1D vector that generated from CNN
    layers.Flatten(),

    #insert dense layer - to decide for classification by using extracted features
    layers.Dense(32, activation="relu"),

    #dropout - to reduce overfitting, need to close randomization neurons during training
    layers.Dropout(0.3),

    #output layer - used sigmoid function because of binary classiification (apple, orange), output neuron
    #probability<= 0.5, class 0, apple
    #probability>0.5, class 1, orange
    layers.Dense(1, activation="sigmoid")
])

#show model structure
model.summary()

**Model Compile**

In [ ]:
#to compile to training model
model.compile(
    #optimizer : adam optimizer to update model weight
    optimizer = "adam",
    #loss function
    #There are binary classification so used binary_crossentropy
    loss = "binary_crossentropy",
    #metric : to watch accuracy during training
    metrics = ["accuracy"]
)

**Early Stopping**

In [ ]:
#watch validation loss during model training
early_stopping = EarlyStopping(
    #monitor validation loss
    monitor = "val_loss",

    #stop training not improve validation loss
    patience = 5, #5 epoch


    #Best Epoch -> No Improvement ->
    #Epoch 1
    #Epoch 2
    #Epoch 3
    #Epoch 4
    #Epoch 5
    #stop


    #better when loss validation loss
    mode = "min",
    #reuse weight when get best validation
    restore_best_weights = True
)

**Model Training**

In [ ]:
#set training epoch times
EPOCHS = 30

#train model
history = model.fit(
    #training dataset
    train_dataset,

    #check validation performance after test
    validation_data = validation_dataset,

    #most 30 epoch times
    epochs = EPOCHS,

    #use early stopping
    callbacks = [early_stopping]
)

**Check actual epochs**

In [ ]:
#store accuracy of each epoch in training history
train_accuracy = history.history["accuracy"]

#get validation accuracy
val_accuracy = history.history["val_accuracy"]

#get training loss
train_loss = history.history["loss"]

#get validation loss
val_loss = history.history["val_loss"]

#count number of training batch times using len()
actual_epochs = len(train_accuracy)

print("Actual Epochs Trained : ", actual_epochs)


**Accuracy Graph**

In [ ]:
#set graph size
plt.figure(figsize=(8,5))

#plot training accuracy
plt.plot(
    range(1, actual_epochs + 1),
    train_accuracy,
    label = "Training Accuracy"
)

#plot validation accuracy
plt.plot(
    range(1, actual_epochs + 1),
    val_accuracy,
    label = "Validation Accuracy"
)

#X-axis Label
plt.xlabel("Epoch")

#T-axis Label
plt.ylabel("Accuracy")

#Graph Title
plt.title("Apple and Orange - Training & Validation Accuracy")

#Legend
plt.legend()

#show grid
plt.grid(True)

#show graph
plt.show()

**Loss Graph**

In [ ]:
#set graph size
plt.figure(figsize=(8,5))

#plot training accuracy
plt.plot(
    range(1, actual_epochs + 1),
    train_loss,
    label = "Training Accuracy"
)

#plot validation accuracy
plt.plot(
    range(1, actual_epochs + 1),
    val_loss,
    label = "Validation Accuracy"
)

#X-axis Label
plt.xlabel("Epoch")

#T-axis Label
plt.ylabel("Loss")

#Graph Title
plt.title("Apple and Orange - Training & Validation Accuracy")

#Legend
plt.legend()

#show grid
plt.grid(True)

#show graph
plt.show()

**Model Preditction**

In [ ]:
#list to store actual label
y_true = []

#list to store predicted label
y_pred = []

#take each batch validation dataset
for images, labels in validation_dataset:
  #predict using model
  predictions_batch = model.predict(
      images,verbose = 0
  )

  #change probability to class label
  predictions_batch = (
    predictions_batch > 0.5
  ).astype(int).flatten()

  #store actual labels
  y_true.extend(labels.numpy())

  #store predicted labels
  y_pred.extend(predictions_batch)

#change list to NumPy Array
y_true = np.array(y_true)
y_pred = np.array(y_pred)

#result
print("Actual Labels : ")
print(y_true)

print("\nPredicted Labels : ")
print(y_pred)


**Confusion Matrix**

In [ ]:
#build confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1] # Ensure all labels are present for proper matrix shape
)

#print matrix
print("=====Confusion Matrix=====")
print(cm)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()


**TP/TN/FP/FN**

In [ ]:
#extract TN, FP, FN, TP from confusion matrix
# Using .ravel() on a 2x2 matrix will unpack in row-major order: TN, FP, FN, TP
TN, FP, FN, TP = cm.ravel()

#print result
print("=====Confusion Matrix Values=====")
print("True Negative (TN) : " , TN)
print("False Positive (FP) : " , FP)
print("False Negative (FN) : " , FN)
print("True Positive (TP) : " , TP)


**Confusion Matrix Visuallization**

In [ ]:
# create object to display confusion matrix graph
disg = ConfusionMatrixDisplay(
    # confusion matrix
    confusion_matrix = cm,

    # class names
    display_labels = class_names
)
# plot matrix
disg.plot()
plt.title("Apple & Orange Confusion Matrix")
plt.show()


**Accuracy**

In [ ]:
# calculate accuracy by comparing actual label and predicted label
accuracy = accuracy_score(
    y_true,
    y_pred
)
# change percentage
print("Accuracy: ", accuracy * 100, "%")


**Precision** : Of all the positive predictions the model made, how many were actually correct?

Formula : Precision = TP/(TP+FP)

 TP (True Positive) = Correctly predicted positive

 FP (False Positive) = Incorrectly predicted positive

In [ ]:
#calculates how many of the predicted positive classes are actually positive.
precision = precision_score(
    y_true,
    y_pred,
    zero_division = 0
)
#In some datasets, zero_dicision = 0 is set to avoid errors if there is no positive prediction.
print("Precision : ", precision * 100, "%")

**Recall**: Of all the actual positive cases, how many did the model correctly identify?

Formula : Recall = TP/(TP+FP)

TP (True Positive) = Correctly Predicted Positive

FN (False Negatice) = Actual positive, but predicted negative

In [ ]:
#calculate how many samples are actually positive that the model can find to be positive.
recall = recall_score(
    y_true,
    y_pred,
    zero_division = 0
)
print("Recall : ", recall * 100, "%")

**Specificity** : Of all the actual negative cases, how many did the model correctly identify?

Formula : Specificity = TN/(TN+FP)

TN (True Negative) = Correctly predicted negative

FP (False Positive) = Incorrectly predicted positive

In [ ]:
#calculate how many of the actual negative samples the model can correctly identify as negative
specificity = ( TN / (TN + TP)
    if (TN + FP) > 0
    else 0
)
print("Specificity : ", specificity * 100, "%")


**F1 Score** : A measure that combines Precision and Recall into one score.

Formula : F1 Score = 2 * (Precision * Recall)/(Precision + Recall) **bold text**

Precision = How many positive predictions were correct

Recall = How many actual positive cases were found

**Higher F1 Score = Better balance between Precision and Recall**

In [ ]:
#metric that combines both Precision and Recall to create a balance
f1 = f1_score(
    y_true,
    y_pred,
    zero_division = 0
)

print("F1 Score : ", f1  * 100, "%")

**FINAL MODEL PERFORMANCE**

In [ ]:
print()
print("=" * 55)
print("MODEL PERFORMANCE")
print("APPLE vs ORANGE")
print("=" * 55)

#Accuracy
print(f"Accuracy : {accuracy * 100:.2f}%")

#PRECISION
print(f"Precision : {precision * 100:.2f}%")

#Recall
print(f"Recall : {recall * 100:.2f}%")

#Specificity
print(f"Specificity : {specificity * 100:.2f}%")

#F1 SCORE
print(f"F1 Score : {f1 * 100:.2f}%")

print("=" * 55)

**Classification Report**

In [ ]:
#Precision, Recall, F1-score and Support are displayed once for each class.
print(
    classification_report(
        #Actual Labels
        y_true,

        #Predicted Labels
        y_pred,
        labels = [0, 1],

        #Class Names
        target_names = class_names,

        #Avoid errors
        zero_division = 0
    )
)

**SAMPLE MODEL INTERPRETATION**

In [ ]:
print("\n========== MODEL INTERPRETATION =========")
#if the accuracy is 90% or highter, it will be shown as high accuracy on the Calidation Set.
if accuracy >= 0.90:
  print("Validation Accuracy is high")
#if the accuracy is below 75% to 90%, it will be shown as Moderate.
elif accuracy >= 0.75:
  print("Validation Accuracy is moderate.")
#if it is below 75%, it will show low accuracy.
else:
  print("Validation Accuracy is low.")

# TRAINING vs VALIDATION GAP
# Compare the Training Accuracy and Validation Accuracy of the last Epoch.

accuracy_gap = abs(train_accuracy[-1] - val_accuracy[-1])

#if the gap is more than 10%, it will show that Overfitting is possible.
if accuracy_gap > 0.10:
  print("Training/Validation Accuracy gap is large.")
  print("Possible Overfitting.")
else:
  print("Training/Validation Accuracy gap is relatively small.")
print("\n========== PROJECT COMPLETED ===========")